# C6-pytorch — Practice p21 — Solution

The NumPy bridge preserves `source`'s float32 dtype, so `t` initially reports `torch.float32`. Elementwise addition promotes the mixed operands to float64, but vector matmul requires matching dtypes and raises; one explicit cast at the boundary produces `t64`, after which both reported operations are unambiguously float64.

In [ ]:
import numpy as np
import torch

source = np.array([1.25, -2.75, 4.5], dtype=np.float32)
t = torch.from_numpy(source)
reference = torch.tensor([0.5, 1.5, -3.0], dtype=torch.float64)

dtype_before = str(t.dtype)
implicit_sum = t + reference
promoted_dtype = str(implicit_sum.dtype)

try:
    t @ reference
except RuntimeError:
    matmul_raises = True
else:
    matmul_raises = False

t64 = t.to(torch.float64)
safe_sum = t64 + reference
safe_dot = t64 @ reference
contract_ok = bool(
    source.dtype == np.float32
    and t64.dtype == torch.float64
    and safe_sum.dtype == torch.float64
    and safe_dot.dtype == torch.float64
    and matmul_raises
)

The safe dot is $1.25(0.5)-2.75(1.5)+4.5(-3)=0.625-4.125-13.5=-17$. The repaired sum is `[1.75, -1.25, 1.5]`, and neither result depends on mixed-dtype promotion.

### Answer check

In [ ]:
assert dtype_before == "torch.float32"
assert promoted_dtype == "torch.float64"
assert matmul_raises is True
assert source.dtype == np.float32
assert np.allclose(
    source, np.array([1.25, -2.75, 4.5], dtype=np.float32), atol=0, rtol=0
)
assert torch.allclose(
    safe_sum,
    torch.tensor([1.75, -1.25, 1.5], dtype=torch.float64),
    atol=0,
    rtol=0,
)
assert torch.allclose(
    safe_dot,
    torch.tensor(-17.0, dtype=torch.float64),
    atol=0,
    rtol=0,
)
assert contract_ok is True